##Ingest circuits csv file
1. read the file using Dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table
    


In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/circuits.csv"
table_name = f"{catalog_name}.{bronze_schema}.circuits"

**Step 1 - Read the CSV file using the dataframe reader API**


normally have to create spark session but databricks already creates it for us 

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'true') #removes first row, keeps header
        .option('inferSchema', 'true') # not suitable for production 
        .load(source_file))

In [0]:
circuits_df.show()

using display

In [0]:
display(circuits_df)


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType([
    StructField('circuitId', StringType()),
    StructField('url', StringType()),
    StructField('circuitName', StringType()),
    StructField('lat', DoubleType()),
    StructField('long', DoubleType()),
    StructField('locality', StringType()),
    StructField('country', StringType()),

])

In [0]:
circuits_df = (
    spark.read
        .format('csv')
        .option('header', 'true') #removes first row, keeps header
 #       .option('inferSchema', 'true') not suitable for production 
        .option('mode', 'FAILFAST')
        .schema(circuits_schema)
        .load(source_file))

**Step 2 - Add the Metadata columns**
  - Source file
  - Ingestion Timestamp


In [0]:

circuits_final_df = add_ingestion_metadata(circuits_df)

In [0]:
display(circuits_final_df)

**Step 3 - Write to bronze delta table**

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
    
)

In [0]:
%sql
SELECT * FROM formula1.bronze.circuits;   

In [0]:
display(spark.table(table_name))